# Conceptual Belief Space Report

This notebook summarizes the paper-style conceptual belief space analysis for the gridworld action commitment experiment. It reads existing outputs from `outputs/conceptual_belief_space/` and `figures/conceptual_belief_space/`.

It does not rerun expensive UMAP fitting, activation manifold fitting, sparse probes, or trajectory analyses by default.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, Image, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'outputs' / 'conceptual_belief_space'
FIG = ROOT / 'figures' / 'conceptual_belief_space'

def show_md(path, max_chars=4000):
    path = Path(path)
    if not path.exists():
        display(Markdown(f'_Missing: `{path.relative_to(ROOT) if path.is_relative_to(ROOT) else path}`_'))
        return
    text = path.read_text()
    if len(text) > max_chars:
        text = text[:max_chars].rstrip() + '\n\n_Excerpt truncated in notebook._'
    display(Markdown(text))

def read_table(path):
    path = Path(path)
    if not path.exists():
        return None
    if path.suffix == '.csv':
        return pd.read_csv(path)
    if path.suffix == '.parquet':
        return pd.read_parquet(path)
    return None

def figure_explanation(path):
    name = Path(path).name
    mapping = {
        'coverage_by_normalized_bin.png': ('normalized progress bin before action mention', 'prefix-row count', 'subset line color', 'each point is a bin count', 'no fitting; coverage diagnostic', 'late bins may have low coverage'),
        'behavior_commitment_before_mention.png': ('normalized progress bin before action mention', 'mean q(final action)', 'subset line color', 'each point is a bin mean over prefixes', 'uses precomputed behavior scores; no fitting', 'pre-action excludes explicit action tokens, not implicit evidence'),
        'true_vs_shuffled_probe_metrics.png': ('control condition', 'top-action accuracy', 'category on x-axis', 'each bar is an aggregate over prefixes', 'belief-level diagnostic in current run; probe controls require Step 7 outputs', 'not activation-alignment evidence by itself'),
        'action_balanced_vs_all_metrics.png': ('all vs balanced condition and metric', 'metric value', 'category on x-axis', 'each bar is an aggregate over selected decisions', 'decision-level balancing; no fitting', 'balanced subsets can increase variance'),
        'cot_length_stratified_metrics.png': ('CoT length stratum', 'mean commitment_to_final', 'category on x-axis', 'each bar is a stratum aggregate', 'no fitting', 'CoT length is observational, not causal')
    }
    return mapping.get(name, ('figure-specific horizontal variable', 'figure-specific vertical variable', 'see title', 'points/lines/bars are outputs from the source analysis', 'read-only display; no new fitting', 'interpret descriptively unless held-out controls support the claim'))

def show_figures(pattern='*.png'):
    paths = sorted(FIG.rglob(pattern))
    if not paths:
        display(Markdown('_No figures found._'))
    for path in paths:
        x, y, color, marks, leakage, caveat = figure_explanation(path)
        display(Markdown(f'### `{path.relative_to(ROOT)}`\n\n- x-axis: {x}\n- y-axis: {y}\n- color: {color}\n- points/lines/bars: {marks}\n- train/test leakage: {leakage}\n- caveat: {caveat}'))
        display(Image(filename=str(path)))

## 1. Motivation: Why Blind PCA Was Insufficient

Blind PCA is diagnostic only. It can reveal high-variance directions, but those directions need not correspond to action concepts. The conceptual belief space pipeline instead anchors behavior and activation geometry using train-only extreme examples for `UP`, `DOWN`, `LEFT`, and `RIGHT`.

Main rule: do not claim that blind PCA shows the activation manifold.

## 2. Dataset and Inputs

This section checks which pipeline artifacts are present.

In [ ]:
reports = [
    OUT / '01_io_audit.md',
    OUT / '02_action_belief_vectors_report.md',
    OUT / '03_extreme_examples_report.md',
    OUT / '04_behavior_manifold_report.md',
    OUT / '05_activation_manifold_report.md',
    OUT / 'geometry_alignment' / '06_geometry_alignment_report.md',
    OUT / 'probes' / '07_probe_report.md',
    OUT / 'trajectories' / '08_trajectory_report.md',
    OUT / 'strict_pre_action' / '09_strict_pre_action_report.md',
    OUT / 'controls' / '10_controls_report.md',
]
availability = pd.DataFrame({'path': [str(p.relative_to(ROOT)) for p in reports], 'available': [p.exists() for p in reports]})
availability

## 3. Behavior Action-belief Construction

Each prefix is represented by an action-belief distribution over `UP`, `DOWN`, `LEFT`, and `RIGHT`, plus transformed versions for geometry and probing.

In [ ]:
beliefs = read_table(OUT / 'action_belief_vectors.parquet')
if beliefs is None:
    display(Markdown('_Missing action belief table._'))
else:
    display(beliefs.head())
    summary = {
        'prefix_rows': len(beliefs),
        'decisions': beliefs['decision_id'].nunique() if 'decision_id' in beliefs else None,
        'trajectories': beliefs['trajectory_id'].nunique() if 'trajectory_id' in beliefs else None,
        'mean_commitment_to_final': beliefs['commitment_to_final'].mean() if 'commitment_to_final' in beliefs else None,
    }
    display(pd.Series(summary).to_frame('value'))

## 4. Train-only Extreme Examples

Extreme examples are selected only from training rows under grouped splits. They anchor conceptual manifolds and centroids without using held-out rows for fitting.

In [ ]:
extremes = read_table(OUT / 'extreme_examples' / 'action_extreme_examples.parquet')
if extremes is None:
    display(Markdown('_Missing extreme-example table._'))
else:
    display(extremes.head())
    display(extremes.groupby(['subset', 'action']).size().rename('n').reset_index().head(20))

## 5. Behavior Conceptual Manifold

Primary method: UMAP fit on train-only behavior action-extreme examples, then transform all prefix belief vectors. PCA is not the main method.

In [ ]:
show_md(OUT / '04_behavior_manifold_report.md')

## 6. Activation Conceptual Manifold

Primary method: activation UMAP fit on the same train extreme row ids used for the behavior manifold. Activation extremes are not selected independently.

In [ ]:
show_md(OUT / '05_activation_manifold_report.md')

## 7. Centroid Distance Matrix Alignment

Behavior-activation alignment should be claimed only if action centroid distance matrices correlate and sparse probes support the same conclusion. With four actions, each centroid matrix comparison has only six pairwise distances.

In [ ]:
show_md(OUT / 'geometry_alignment' / '06_geometry_alignment_report.md')

## 8. Sparse Probes

Sparse probes map high-dimensional activation features to behavior belief vectors under grouped splits. UMAP/PCA coordinates are not probe inputs. Negative R2 should be reported honestly.

In [ ]:
show_md(OUT / 'probes' / '07_probe_report.md')

## 9. Trajectory Analysis

Trajectory analysis compares decision paths through behavior and activation conceptual spaces. Nearest-centroid basins are descriptive and should not be described as causal mechanisms.

In [ ]:
show_md(OUT / 'trajectories' / '08_trajectory_report.md')

## 10. Strict Pre-action Analysis

Pre-action means no explicit lexical action token has appeared in the CoT. It does not mean there is no implicit coordinate or path evidence.

In [ ]:
show_md(OUT / 'strict_pre_action' / '09_strict_pre_action_report.md')
strict_metrics = read_table(OUT / 'strict_pre_action' / 'strict_pre_action_metrics.csv')
if strict_metrics is not None:
    display(strict_metrics.head(20))

## 11. Robustness Controls

Controls include shuffled labels, shuffled behavior q vectors, action-balanced subsets, CoT-length stratification, split sensitivity, representation ablations, and extreme-selection ablations.

In [ ]:
show_md(OUT / 'controls' / '10_controls_report.md')
controls = read_table(OUT / 'controls' / 'control_metrics.csv')
if controls is not None:
    display(controls.groupby('control_family').size().rename('n').reset_index())

## Figure Guide

For each figure, the notebook lists the x-axis, y-axis, color encoding, what each mark represents, train/test leakage status, and caveat.

In [ ]:
show_figures()

## 12. Interpretation and Limitations

- Do not claim that blind PCA shows the activation manifold.
- Do not claim behavior-activation alignment unless distance-matrix correlations and probe metrics support it.
- If alignment is weak or missing, say it is weak or missing.
- Four actions give only six pairwise centroid distances.
- CoT-token activations are distinct from final JSON decision-readout activations.
- Observational curves are not causal evidence.

## What Changed Relative to the Previous PCA Analysis

- Blind PCA is now diagnostic only.
- Main geometry is anchored by train-only action-extreme examples.
- Behavior and activation spaces are compared by centroid distance matrices and probes, not visual similarity alone.
- Strict pre-action subsets are analyzed separately.
- Grouped train/test splits are required for fitted analyses.
- CoT-token activations are distinguished from final JSON decision-readout activations.

## 13. Next Experiments

1. Install `umap-learn` and rerun Steps 4 and 5.
2. Rerun Step 6 for centroid distance matrix alignment.
3. Rerun Step 7 for sparse probes across layers and representations.
4. Rerun Steps 8-10 after probe and geometry outputs exist.
5. Add final JSON decision-readout activations and compare them with CoT-token activations.
6. Run causal interventions only after descriptive geometry/probe evidence is stable.